# Software-as-a-Graph (SaG) — JSS Journal Paper Reproducibility Suite
### *Heterogeneous Graph Learning for Pre-Deployment Reliability and Dependability Analysis of Complex Distributed Systems*
**Journal of Systems and Software (JSS) — Special Issue VSI:AI4MSS**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook provides a cloud-ready, GPU-accelerated environment to train all GNN models and reproduce the empirical results, tables, and figures for the JSS submission.

---

### Hardware Accelerator Setup (Colab GPU)
1. In the Colab menu, go to **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (free tier) or **A100 / V100** (Colab Pro).
3. Click **Save**.

## 1. Workspace & Drive Setup
Mount Google Drive (recommended to persist outputs) or clone the repository directly.

In [ ]:
# --- Method A: Mount Google Drive (Recommended) ---
from google.colab import drive
import os

drive.mount('/content/drive')

# Option A1: If entire project is zipped in Drive:
# !unzip -q /content/drive/MyDrive/SoftwareAsAGraph.zip -d /content/SoftwareAsAGraph
# Option A2: If using loso_cache.tar.gz from Drive:
# !mkdir -p /content/SoftwareAsAGraph/output && tar -xzf /content/drive/MyDrive/loso_cache.tar.gz -C /content/SoftwareAsAGraph/output/

# --- Method B: Clone from GitHub ---
# !git clone https://github.com/<your-username>/SoftwareAsAGraph.git /content/SoftwareAsAGraph
# If loso_cache was gitignored, upload output/loso_cache.tar.gz to Colab and unpack:
# !mkdir -p /content/SoftwareAsAGraph/output && tar -xzf /content/loso_cache.tar.gz -C /content/SoftwareAsAGraph/output/

REPO_DIR = "/content/SoftwareAsAGraph"
if not os.path.exists(REPO_DIR):
    REPO_DIR = "/content"

%cd {REPO_DIR}
%set_env PYTHONPATH=.


## 2. Hardware Verification & Dependencies
Verify GPU availability and install PyTorch Geometric along with the repository package.

In [ ]:
# Inspect GPU hardware
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Device Count:", torch.cuda.device_count())
else:
    print("⚠️ GPU not detected. Please enable GPU in Runtime > Change runtime type.")

In [ ]:
# Install PyTorch Geometric and SaaG core package
!pip install -q torch-geometric
!pip install -q -e .
print("✓ Dependencies installed successfully.")

## 3. Block 0: QoS Pipeline Audit (Go/No-Go Gate)
Runs the W1 QoS pipeline audit gate (`tests/test_qos_pipeline_audit.py` and `tests/test_baselines.py`) to verify topological formulation integrity and baseline correctness before training.

In [ ]:
!pytest tests/test_qos_pipeline_audit.py tests/test_baselines.py -v --tb=short -q

## 4. Smoke-Test (Fast Sanity Check)
Runs a lightweight training sweep (3 scenarios, 50 epochs, 2 seeds) taking ~1–2 minutes on GPU to verify end-to-end execution before running the full 300-epoch matrix.

In [ ]:
!python reproduce/main_table.py \
    --scenarios atm_system av_system iot_smart_city_system \
    --seeds 42 123 \
    --epochs 50 \
    --output results/smoke_main_table.json \
    --resume

## 5. Experiment 1: In-Distribution Evaluation (JSS Table 6 & Table 7)
Trains the full 7 scenarios × 6 model variants × 5 random seeds = 210 evaluation cells:
- **Structural Baselines**: `Topo`, `Topo-QoS`
- **Homogeneous GNNs**: `GAT`, `GAT-QoS`
- **Heterogeneous GNNs**: `HGT` (masked QoS ablation), `HGT-QoS` (proposed model)

*Resilience*: Uses `--resume` so if the Colab session disconnects, re-running skips completed runs.

In [ ]:
# Train full in-distribution matrix
!python reproduce/main_table.py \
    --output results/main_table.json \
    --seeds 42 123 456 789 2024 \
    --epochs 300 \
    --resume

In [ ]:
# Render LaTeX and Markdown tables (results/table3_main_results.tex & .md)
!python reproduce/render_table.py \
    --table3 results/main_table.json \
    --output-dir results

# Display rendered markdown table inline
from IPython.display import display, Markdown
if os.path.exists("results/table3_main_results.md"):
    with open("results/table3_main_results.md") as f:
        display(Markdown(f.read()))

## 6. Experiment 2: Inductive Cross-Domain Generalization (LOSO, JSS Table 8)
Evaluates cross-domain generalization via Leave-One-Scenario-Out (LOSO) cross-validation and computes Wilcoxon signed-rank tests.

In [ ]:
# Train LOSO across all variants
!python reproduce/loso_all_variants.py \
    --cache-dir output/loso_cache \
    --epochs 300 \
    --output results/loso_all_variants.json \
    --resume

# Render Table 8 LaTeX output
!python reproduce/render_table.py \
    --table4 results/loso_all_variants.json \
    --output-dir results

# Calculate statistical significance & p-values
!python reproduce/loso_significance.py \
    --input results/loso_all_variants.json \
    --output results/loso_significance.json

In [ ]:
# Display summary of LOSO significance
import json
if os.path.exists("results/loso_significance.json"):
    with open("results/loso_significance.json") as f:
        sig = json.load(f)
    print("=== LOSO Wilcoxon Significance Results ===")
    print(json.dumps(sig, indent=2))

## 7. Experiment 3: In-Domain Per-Domain K-Fold Evaluation (JSS Table 9)
Trains 5 variants across $k$ folds per domain scenario (opt-in validation protocol).

In [ ]:
# Optional: run per-domain k-fold cross-validation
!python reproduce/kfold_all_variants.py \
    --cache-dir output/loso_cache \
    --epochs 300 \
    --output results/kfold_all_variants.json \
    --resume

!python reproduce/render_table.py \
    --table-kfold results/kfold_all_variants.json \
    --output-dir results

## 8. Case Study & Figures (Attention Subgraphs & JSS Figures 3–5)
Generates the publication figures:
- **Figure S2 / Figure 5**: ATM Case Study HGT Attention Subgraph.
- **Figure 3**: Results-at-a-glance (LOSO Spearman $\rho$, F1@K, Oracle agreement).
- **Figure 4**: Stratified per-node-type $\rho$.

In [ ]:
# Extract attention weights for ATM scenario and render subgraph
!python reproduce/extract_attention.py \
    --scenario atm_system \
    --output-dir output/atm_case_study

!python reproduce/render_attention_subgraph.py \
    --input output/atm_case_study/attention_weights.json \
    --output output/atm_case_study/attention_subgraph

# Render results-at-a-glance figure
!python reproduce/render_results_figure.py
!python reproduce/render_stratified_figure.py --source auto --output results/figure4_stratified_rho

In [ ]:
# Display rendered figures
from IPython.display import Image, display
from pathlib import Path

figures = [
    Path("output/atm_case_study/attention_subgraph.png"),
    Path("docs/research/jss/latex/figures/Figure_3.png"),
    Path("results/figure4_stratified_rho.png")
]

for fig in figures:
    if fig.exists():
        print(f"\nFigure: {fig}")
        display(Image(filename=str(fig)))

## 9. Backup Checkpoints & Results to Google Drive
Persists trained model checkpoints (`output/gnn_checkpoints/`) and output tables/figures (`results/`) to Google Drive.

In [ ]:
from datetime import datetime

backup_dir = f"/content/drive/MyDrive/SaG_JSS_Results_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(backup_dir, exist_ok=True)

!cp -r results {backup_dir}/
if os.path.exists("output/gnn_checkpoints"):
    !cp -r output/gnn_checkpoints {backup_dir}/
if os.path.exists("output/atm_case_study"):
    !cp -r output/atm_case_study {backup_dir}/

print(f"✓ Experimental artifacts backed up to: {backup_dir}")